# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10`. `s["b"]` обращается к элементу по метке индекса `"b"` и возвращает 20, а `s.iloc[0]` берёт первый элемент по позиции и возвращает 10.
2. Получится Series:
   - Алматы — 5
   - Астана — NaN
   - Шымкент — 3

   Pandas складывает Series по меткам индексов, а не по позициям. Для «Астана» нет значения в `a`, поэтому результат — `NaN`.
3. `(2, 2)`. Условие `x > 1` оставляет две строки (со значениями x=2 и x=3), а столбцов остаётся два.
4. `Series DataFrame`. `df["x"]` возвращает один столбец как Series, а `df[["x"]]` — таблицу DataFrame с одним столбцом.
5. `[1, 2] 5`. После фильтрации сохраняются исходные индексы строк 1 и 2. `f.iloc[0, 1]` берёт значение из первой строки и второго столбца отфильтрованной таблицы — 5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [4]:
# 1. Паспорт таблицы
print("Размер таблицы (строки, столбцы):", df.shape)

print("\\nТипы столбцов:")
print(df.dtypes)

print("\\nПропуски в delivery_days:", df["delivery_days"].isna().sum())

print("\\nКоличество уникальных значений:")
for col in ["city", "category", "status", "customer_id"]:
    print(col, ":", df[col].nunique())

print("\\nВывод для менеджера:")
print("1. В city могут быть разные написания одного города. Это разделяет один город на несколько категорий и искажает сравнение выручки.")
print("2. В delivery_days есть пропуски. По таким заказам нельзя корректно оценить сроки доставки; отчёт по доставке будет неполным.")
print("3. В данных могут быть тестовые заказы (customer_id = TEST) и повторяющиеся order_id. Если их не проверить, выручка и число заказов могут быть завышены или искажены.")

Размер таблицы (строки, столбцы): (5463, 8)
\nТипы столбцов:
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
\nПропуски в delivery_days: 160
\nКоличество уникальных значений:
city : 14
category : 5
status : 2
customer_id : 2003
\nВывод для менеджера:
1. В city могут быть разные написания одного города. Это разделяет один город на несколько категорий и искажает сравнение выручки.
2. В delivery_days есть пропуски. По таким заказам нельзя корректно оценить сроки доставки; отчёт по доставке будет неполным.
3. В данных могут быть тестовые заказы (customer_id = TEST) и повторяющиеся order_id. Если их не проверить, выручка и число заказов могут быть завышены или искажены.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [5]:
# 2. Типичный заказ
valid_orders = df[paid & not_test]["amount"]

print("Количество оплаченных заказов без TEST:", valid_orders.count())
print("Среднее:", round(valid_orders.mean(), 2))
print("Медиана:", round(valid_orders.median(), 2))
print("25-й перцентиль:", round(valid_orders.quantile(0.25), 2))
print("75-й перцентиль:", round(valid_orders.quantile(0.75), 2))

print("\\nВывод для менеджера:")
print("Медиана лучше описывает типичный чек, если в данных есть очень крупные заказы: такие значения сильно влияют на среднее, но почти не влияют на медиану.")

Количество оплаченных заказов без TEST: 4901
Среднее: 14672.39
Медиана: 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0
\nВывод для менеджера:
Медиана лучше описывает типичный чек, если в данных есть очень крупные заказы: такие значения сильно влияют на среднее, но почти не влияют на медиану.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [6]:
# 3. Отмены по городам
cancelled = df["status"] == "отменён"
almaty_names = ["Алматы", "алматы", "Almaty"]

def cancellation_share(mask):
    total = mask.sum()
    cancelled_count = (mask & cancelled).sum()
    share = cancelled_count / total if total else float("nan")
    return cancelled_count, total, share

all_result = cancellation_share(pd.Series(True, index=df.index))
sh_result = cancellation_share(df["city"].isin(SH))
al_result = cancellation_share(df["city"].isin(almaty_names))

print("Все заказы: отменено", all_result[0], "из", all_result[1],
      "| доля:", round(all_result[2] * 100, 2), "%")
print("Шымкент: отменено", sh_result[0], "из", sh_result[1],
      "| доля:", round(sh_result[2] * 100, 2), "%")
print("Алматы: отменено", al_result[0], "из", al_result[1],
      "| доля:", round(al_result[2] * 100, 2), "%")

if sh_result[2] > al_result[2]:
    higher, lower, city_name = sh_result[2], al_result[2], "Шымкент"
else:
    higher, lower, city_name = al_result[2], sh_result[2], "Алматы"

print("\\nВывод для менеджера:")
if lower > 0:
    print(city_name, "имеет более высокую долю отмен — примерно в",
          round(higher / lower, 2), "раза.")
else:
    print("Сравнить доли в разах нельзя: в одном из городов доля отмен равна нулю.")

Все заказы: отменено 559 из 5463 | доля: 10.23 %
Шымкент: отменено 164 из 804 | доля: 20.4 %
Алматы: отменено 167 из 2060 | доля: 8.11 %
\nВывод для менеджера:
Шымкент имеет более высокую долю отмен — примерно в 2.52 раза.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [7]:
# 4. Октябрь против сентября по двум городам
cities = {
    "Шымкент": SH,
    "Алматы": almaty_names
}

for city_name, city_variants in cities.items():
    city_mask = df["city"].isin(city_variants)
    september_revenue = df.loc[sep & paid & not_test & city_mask, "amount"].sum()
    october_revenue = df.loc[octo & paid & not_test & city_mask, "amount"].sum()

    if september_revenue != 0:
        change = (october_revenue - september_revenue) / september_revenue * 100
    else:
        change = float("nan")

    print(city_name)
    print("  Выручка сентября:", september_revenue)
    print("  Выручка октября:", october_revenue)
    print("  Изменение:", round(change, 2), "%")

    if september_revenue != 0:
        print("  Изменение в тенге:", october_revenue - september_revenue)

print("\\nВывод для менеджера:")
print("Отрицательное изменение означает падение выручки. Сравните проценты двух городов: большее по модулю отрицательное значение показывает более сильное падение. Дубли order_id в этом задании намеренно не удаляются.")

Шымкент
  Выручка сентября: 4949800.0
  Выручка октября: 4044010.0
  Изменение: -18.3 %
  Изменение в тенге: -905790.0
Алматы
  Выручка сентября: 14510240.0
  Выручка октября: 13989050.0
  Изменение: -3.59 %
  Изменение в тенге: -521190.0
\nВывод для менеджера:
Отрицательное изменение означает падение выручки. Сравните проценты двух городов: большее по модулю отрицательное значение показывает более сильное падение. Дубли order_id в этом задании намеренно не удаляются.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [8]:
# 5. Исправленный код
# Ошибка 1: каждое сравнение нужно заключить в скобки.
# Ошибка 2: нужно объединять условия через &, а не пытаться сравнить строку
# "Шымкент" с результатом выражения с приоритетом операторов.

cancel_sh = df[
    (df["city"].isin(SH)) &
    (df["status"] == "отменён")
]
print("Отменённые заказы Шымкента:", len(cancel_sh))

Отменённые заказы Шымкента: 164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [9]:
# 6. Самые частые клиенты
customer_counts = df["customer_id"].value_counts()
top_customer = customer_counts.index[0]
top_count = customer_counts.iloc[0]

print("Самый активный клиент:", top_customer)
print("Количество его заказов:", top_count)

top_customer_orders = df.loc[
    df["customer_id"] == top_customer,
    ["order_date", "city", "amount", "status", "order_id"]
]
print("\\nВсе заказы этого клиента:")
print(top_customer_orders.to_string(index=False))

duplicate_order_ids = top_customer_orders["order_id"].duplicated(keep=False)
print("\\nПовторяющиеся order_id:")
print(top_customer_orders.loc[duplicate_order_ids].to_string(index=False))

if duplicate_order_ids.any():
    print("\\nВывод для менеджера:")
    print("У самого активного клиента есть повторяющиеся order_id. Это может быть повторная запись одного заказа; при подсчёте выручки без проверки дублей сумма может быть завышена.")
else:
    print("\\nВывод для менеджера:")
    print("У самого активного клиента повторяющихся order_id не обнаружено. Однако это не доказывает, что дублей нет у остальных клиентов.")

Самый активный клиент: C01818
Количество его заказов: 11
\nВсе заказы этого клиента:
      order_date      city  amount  status  order_id
2026-09-01 15:00   Шымкент  7350.0 оплачен    101125
2026-09-09 18:00   Шымкент 45270.0 оплачен    100096
2026-09-15 21:00    Алматы  8890.0 оплачен    101504
2026-09-15 21:00    Алматы  8890.0 оплачен    101504
2026-09-24 18:00    Астана  1490.0 оплачен    101860
2026-09-29 09:00    Астана 22170.0 оплачен    101462
2026-09-30 21:00    Алматы 20180.0 оплачен    101495
2026-10-08 14:00 Караганда  4330.0 оплачен    103416
2026-10-18 08:00    Астана 13600.0 оплачен    102644
2026-10-21 19:00    Алматы 21990.0 оплачен    104001
2026-10-23 18:00    Алматы 11140.0 оплачен    104572
\nПовторяющиеся order_id:
      order_date   city  amount  status  order_id
2026-09-15 21:00 Алматы  8890.0 оплачен    101504
2026-09-15 21:00 Алматы  8890.0 оплачен    101504
\nВывод для менеджера:
У самого активного клиента есть повторяющиеся order_id. Это может быть повторная

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [10]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [11]:
# 8